# OEV Colab training

Runtime > Change runtime type > T4 GPU. Then run cells top to bottom.

The backbone cells fine-tune DeBERTa-v3-base (~740MB download first time). Expect roughly 2 hours for the typed-decisions run on a free T4.

In [ ]:
%cd /content
!rm -rf oev
!git clone https://github.com/divyanshudhruv/oev.git
%cd /content/oev
!pip install -q -e ".[dev,data,backbone]"
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!python -m pytest -q   # 33 passing, 4 skipped (backbone tests)

## 1. Synthetic domain (from-scratch char model)

In [ ]:
!python -m oev.data_gen

In [ ]:
!python -m oev.train --preset tiny --epochs 4 --batch-size 256

In [ ]:
!python -m oev.evaluate --checkpoint checkpoints/oev-tiny.pt

## 2. Public benchmarks (from-scratch char model)

AG News and emotion converted to OEV format. Test splits stay sealed from training.

In [ ]:
!python -m oev.convert

In [ ]:
!python -m oev.train --preset tiny --epochs 2 --batch-size 128 --data-dir data/ag_news --out checkpoints_ag

In [ ]:
!python -m oev.benchmark --checkpoint checkpoints_ag/oev-tiny.pt --data-dir data/ag_news

## 3. Backbone fine-tune (DeBERTa-v3-base)

DeBERTa-v3-base already knows English; fine-tuning teaches only the decision task. Expect 0.90+ on AG News vs the ~0.29 from-scratch result.

Batch size 16 fits a T4 at max length 512; the env var set in the first cell avoids fragmentation.

In [ ]:
!python -m oev.train --backbone microsoft/deberta-v3-base --epochs 2 --batch-size 16 --data-dir data/ag_news --out checkpoints_bb_base
!python -m oev.benchmark --checkpoint checkpoints_bb_base/oev-tiny.pt --data-dir data/ag_news

In [ ]:
!python -m oev.train --backbone microsoft/deberta-v3-base --epochs 2 --batch-size 16 --data-dir data/emotion --out checkpoints_bb_em_base
!python -m oev.benchmark --checkpoint checkpoints_bb_em_base/oev-tiny.pt --data-dir data/emotion

## 4. Typed-decisions benchmark (the headline fight)

Public dataset: `LocalLLaMA/typed-decisions` - 1,200 train cases / 6,000 decisions, 400 test cases / 2,000 decisions across 4 workflows. Laya's fine-tuned 421M model scores 0.766 here; Jev 0.727; teacher self-agreement 0.735; majority class 0.461.

Long sequences (max length 768) need a smaller batch. This is the multi-hour cell.

In [ ]:
!python -m oev.convert_typed

In [ ]:
!python -m oev.train --backbone microsoft/deberta-v3-base --epochs 4 --batch-size 8 --max-len 768 --data-dir data/typed --out checkpoints_td5

In [ ]:
!python -m oev.benchmark_ext --checkpoint checkpoints_td5/oev-tiny.pt --data-dir data/typed

## 5. RLCD + ensemble (adds roughly 2 points)

In [ ]:
!python -m oev.rlcd --checkpoint checkpoints_td5/oev-tiny.pt --data-dir data/typed --epochs 2 --batch-size 8 --out checkpoints_rlcd

In [ ]:
!python -m oev.ensemble --ckpts checkpoints_td5/oev-tiny.pt,checkpoints_rlcd/oev-tiny.pt,checkpoints_rlcd_soup/oev-tiny.pt --data-dir data/typed

## 6. Try the decide() API

In [ ]:
from oev.infer import OEV
agent = OEV("checkpoints_bb_base/oev-tiny.pt", device="cuda")
print(agent.decide("Wall St. Bears Claw Back Into the Black (Reuters)", {
    "topic": {"type": "choice", "options": ["world", "sports", "business", "sci/tech"], "instructions": "Which topic does this news article belong to?"},
}))
print(agent.decide("We were charged twice for the same order.", {
    "department": {"type": "choice", "options": ["billing", "technical", "sales", "other"]},
    "refund_requested": {"type": "noul"},
    "severity": {"type": "score", "levels": [1, 2, 3, 4, 5]},
}))

## 7. Save results to Drive (recommended)

Mount Drive first, then checkpoints survive Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/oev
!cp -r checkpoints* /content/drive/MyDrive/oev/